## Evaluate linear probe for classification

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path
import pandas as pd

from dual_ifm.classification.eval_finetune import get_all_metrics, load_eval

# Run in parent dir
cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    os.chdir(cwd.parent)

pd.options.display.float_format = "{:,.3f}".format

In [ ]:
project_dir = Path.cwd()
checkpoints_dir = project_dir.joinpath('checkpoints')

In [ ]:
backbone_names = ['ResNet50', 'BagNet33']
weight_names = ['ImageNet', 'SimCLR', 't-SimCNE', 't-SimCNEx'] #
# dataset_names = ['EyePACS', 'AREDS', 'UKB'] # In pretraining
dataset_names = ['APTOS', 'DeepDRiD', 'IDRiD', 'Messidor', 'Glaucoma', 'PAPILA', 'FIVES']

df_rows = []
for dataset_name in dataset_names:
    if dataset_name in ['AREDS']:
        feature_name = 'amd'
    elif dataset_name in ['Glaucoma', 'PAPILA']:
        feature_name = 'glaucoma'
    elif dataset_name in ['FIVES']:
        feature_name = 'disease'
    else:
        feature_name = 'dr'

    for backbone_name in backbone_names:
        for weight_name in weight_names:
            # Experiment name
            prefix = f'{weight_name.replace('-', '').lower()}{backbone_name[0].lower()}'
            experiment_name = f'clflin_{prefix}_{dataset_name.lower()}_{feature_name}_256'

            checkpoint_dir = checkpoints_dir.joinpath(experiment_name)
            if os.path.exists(checkpoint_dir.joinpath(experiment_name + '.json')):
                preds, probs, targets, stats, wandb_id = load_eval(checkpoint_dir, experiment_name, load_stats=False)
                auroc, auprc, acc, kappa = get_all_metrics(preds, probs, targets)
                row = {'Model': f'{backbone_name} {weight_name}', 'Dataset': dataset_name, 'AUROC': auroc, 'AUPRC': auprc, 'Balanced Accuracy': acc, 'Kappa': kappa}
                df_rows.append(row)

df = pd.DataFrame(df_rows)

In [ ]:
df_groupby = df.groupby(by=['Dataset', 'Model']).agg('mean')
df_groupby

In [ ]:
# Display one metric
df_metric = df_groupby['AUROC'].copy().unstack('Model')
table_metric = df_metric.reindex(dataset_names)
table_metric

In [ ]:
print(table_metric.to_csv(sep='\t'))

In [ ]:
latex_table = table_metric.to_latex(
    float_format='%.3f',
    multicolumn=True,
    multirow=True,
    caption='caption',
    label='tab:label',
)

print(latex_table)